In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
import scipy.stats as stats
from scipy.integrate import quad
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import utilities.plot_settings
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D

def update(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)
    handle.set_markersize(3)

In [ ]:
# Characteristic neutron star radius in [cm].
NS_radius = 6.e8 #7.e8

# Characteristic neutron star mass in solar masses.
NS_mass = 1. * const.M_SUN

# Dimensionless coefficients k_0, k_1, k_2 for a force-free magnetosphere
# taken from Spitkovsky (2006) and Philippov et al. (2014).
# For comparison, in vacuum k_0 = 0 and k_1 = k_2 = 2/3.
k_coefficients = [1.0, 1.0, 1.0]

# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * NS_mass * NS_radius ** 2

# Auxiliary quantity beta as defined in eq. (72) of Pons & Vigano (2019).
beta = 1./4. * NS_radius ** 6 / (NS_inertia * const.C ** 3)
#beta = np.pi ** 2 * NS_radius ** 6 / (NS_inertia * const.c ** 3)
print(beta)

# Assume an inclination angle in [rad].
chi = 0.

beta_1 = beta * (k_coefficients[0] + k_coefficients[1] * np.sin(chi) ** 2)
print(beta_1)

In [ ]:
def period_derivative_B(P: float, B: float) -> float:
    """
    This function determines the change in the rotation period of a pulsar. It is taken
    from eq. (70) of Pons & Vigano (2019). For more details see, e.g., Spitkovsky (2006)
    or Philippov et al. (2014), who determine the coefficients k_0, k_1, k_2 (defined in
    the configuration file) for a pulsar embedded in a force-free and resistive magnetosphere
    from numerical simulations. Note that all three input parameters are time-dependent.

    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        B (float): value of the dipolar component of the magnetic field at the
        magnetic pole for a simulated neutron star, measured in [G].      

    Returns:
        (float): period derivative of a simulated pulsar in [s/s].
    """

    # Period derivative.
    P_deriv = 4 * np.pi**2 * beta_1 * B ** 2 / P

    return P_deriv

def period_derivative_tage(P: float, tage: float) -> float:
    """
    Pdot as a function of period with varying pulsar age assuming P >> P0 and constant magnetic field.
    
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        tage (float): characteristic age in [yr].      

    Returns:
        (float): period derivative of a simulated pulsar in [s/s].
    """
    tage = tage * const.YR_TO_S
    
    P_deriv = P/(2.*tage)
    
    return P_deriv

def period_derivative_Erot(P: float, Erot: float) -> float:
    """
    Pdot as a function of period with varying pulsar rotational energy derivative.
    
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Erot (float): rotational energy derivative [erg/s^2].      

    Returns:
        (float): period derivative of a simulated pulsar in [s/s].
    """
    
    P_deriv = (Erot * P**3) / ((2.*np.pi)**2 * NS_inertia)
    
    return P_deriv

def B_from_timing(P: float, Pdot: float) -> float:
    """
    B field estimated from timing properties. 
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): value of the dipolar component of the magnetic field at the
        magnetic pole for a simulated neutron star, measured in [G].
    """

    # Period derivative.
    B = np.sqrt(P * Pdot / (4 * np.pi**2 * beta_1))

    return B

def Edot_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar rotational power loss from timing properties. 
    
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): characteristic age in [yr].     
    """

    # Period derivative.
    Erot_dot = (2.*np.pi)**2 * NS_inertia * Pdot / P**3

    return Erot_dot

In [ ]:
data_i_wd1 = pd.read_pickle(
    "../toremove/nanda_experiment/WD_exp1_1e7/initial_population.pkl.gz",
    compression="gzip",
)
data_i_wd1.head()

In [ ]:
P_i_wd1 = data_i_wd1["P"]["[s]"].to_numpy()
P_dot_i_wd1 = data_i_wd1["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
data_f_wd1 = pd.read_pickle(
    "../toremove/nanda_experiment/WD_exp1_1e7/final_population.pkl.gz",
    compression="gzip",
)
data_f_wd1.head()

In [ ]:
P_f_wd1 = data_f_wd1["P"]["[s]"].to_numpy()
P_dot_f_wd1 = data_f_wd1["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol_wd1 = data_f_wd1["L_radio_bol"]["[erg s^-1]"].to_numpy()
B_f_wd1 = data_f_wd1["B"]["[G]"].to_numpy()

intercept_radio_wd1 = data_f_wd1[('intercepted_radio',' ')] == 1

In [ ]:
P_min = 1.e-2
P_max = 1.e10
Pdot_min = 1.e-32
Pdot_max = 1.e-4

log_P_edges = np.linspace(np.log10(P_min), np.log10(P_max), 71)
log_P_centers = 0.5 * (log_P_edges[1:] + log_P_edges[:-1])
P_edges = 10**log_P_edges
P_centers = 10**log_P_centers

log_Pdot_edges = np.linspace(np.log10(Pdot_min), np.log10(Pdot_max), 71)
log_Pdot_centers = 0.5 * (log_Pdot_edges[1:] + log_Pdot_edges[:-1])
Pdot_edges = 10**log_Pdot_edges
Pdot_centers = 10**log_Pdot_centers

P_grid, Pdot_grid = np.meshgrid(P_centers, Pdot_centers, indexing='ij')

B_timing = B_from_timing(P_grid, Pdot_grid)
Edot_timing = Edot_from_timing(P_grid, Pdot_grid)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [1, 3], "hspace": 0})

plt.xscale('log') 
plt.yscale('log') 

# Plot the distribution of P_i_1 at the top
P_bins = np.logspace(-2, 10, 31)

ax1.set_ylabel("# WDs")
ax1.set_xscale('log') 
ax1.set_yscale('log') 
ax1.tick_params(axis="x", labelbottom=False)
ax1.set_yticks([1e2, 1e4, 1e6])

ax1.hist(
    P_i_wd1, 
    bins=P_bins,
    histtype="step",
    linewidth=4,
    color="grey",
    #density=True,
)
ax1.hist(
    P_f_wd1, 
    bins=P_bins,
    histtype="step",
    linewidth=4,
    color="tab:pink",
    #density=True,
)
ax1.hist(
    P_f_wd1[intercept_radio_wd1], 
    bins=P_bins,
    histtype="step",
    linewidth=4,
    color="indigo",
    #density=True,
)

ax2.set_xlabel(r"$P$ [s]")
ax2.set_ylabel(r"$\dot{P}$")
ax2.set_xscale('log') 
ax2.set_yscale('log') 
ax2.set_xlim(P_min,P_max)
ax2.set_ylim(Pdot_min,Pdot_max)

contour_B = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(B_timing), 
    levels = np.array([2.,4.,6.,8.,10.,12.]), 
    colors='black',
    linestyles='dashed',
    alpha=0.7
    #interpolation='none'
)
fmt = {}
strs = ['$10^{2}$ G', '$10^{4}$ G', '$10^{6}$ G', '$10^{8}$ G', '$10^{10}$ G', '$10^{12}$ G']
for l,s in zip( contour_B.levels, strs ):
    fmt[l] = rf"{s}"
manual_locations = [(5.e-2, 1e-22), (5.e-2, 1e-19), (5.e-2, 1e-15), (5.e-2, 1e-10), (5.e-2, 1e-7), (10., 1e-6)]
ax2.clabel(
    contour_B, 
    contour_B.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)

contour_Edot = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(Edot_timing),
    levels = np.array([5.,15.,25.,35.]), 
    colors='black',
    linestyles='dashed',
    alpha=0.7
    #interpolation='none'
)
fmt = {}
strs = ['$10^{5}$ erg s$^{-1}$', '$10^{15}$ erg s$^{-1}$', '$10^{25}$ erg s$^{-1}$', '$10^{35}$ erg s$^{-1}$']
for l,s in zip( contour_Edot.levels, strs ):
    fmt[l] = rf"{s}"
manual_locations = [(1e9, 1e-20), (1e9, 1e-12), (1e7, 1e-7), (1e3, 1e-7)]
ax2.clabel(
    contour_Edot, 
    contour_Edot.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)

ax2.plot(
    P_i_wd1,
    P_dot_i_wd1,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=3,
    alpha=0.1,
    rasterized=True,
    label = 'Initial population WD1'
)

ax2.plot(
    P_f_wd1,
    P_dot_f_wd1,
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=3,
    alpha=0.1,
    rasterized=True,
    label = 'Final population WD1'
)
ax2.plot(
    P_f_wd1[intercept_radio_wd1],
    P_dot_f_wd1[intercept_radio_wd1],
    linestyle="None",
    marker="o",
    #mfc='none',
    color="indigo",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los WD1'
)

#plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20,markerscale=5)
plt.legend(frameon=False, loc=3, fontsize=20, markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

plt.savefig("fig4_WD_ppdot.pdf")
plt.show()

In [ ]:
m = 1. * const.M_SUN
R = 6.e8
I = (2/5)*m*R**2
Erot = ((2*np.pi)**2)*NS_inertia*(P_dot_f_wd1/P_f_wd1**3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))


ax.plot(
    P_f_wd1,
    Erot,
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=3,
    alpha=0.1,
    rasterized=True,
    label = 'Final population WD1'
)

ax.plot(
    P_f_wd1[intercept_radio_wd1],
    Erot[intercept_radio_wd1],
    linestyle="None",
    marker="o",
    color="indigo",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercepting our los WD1'
)


#plt.axvline(10**(-0.7))
ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$\dot{E}_{\rm rot}$")
ax.set_xlim(P_min,P_max)
plt.xscale('log') 
plt.yscale('log') 

#plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20,markerscale=5)
plt.legend(frameon=False, loc=0, fontsize=20,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

plt.grid()
plt.savefig("fig4_WD_Erot.pdf")
plt.show()